# Notebook 0 — Exécuter le pipeline *cellule après cellule*

## Objectif

Enchaîner **NB1 → NB2 → NB3 → NB4** de façon contrôlée : chaque cellule = une étape.

Vous pouvez :
- exécuter **tout** (`Run All`) ;
- ou avancer **étape par étape** (recommandé la première fois).

Guide : `00_PIPELINE_PAS_A_PAS.md`

## Stratégie

| Étape | Action par défaut | Alternative longue |
|---|---|---|
| 1 Acquisition | script `download_opensensemap_thp.py` si raw absent | Notebook 1 |
| 2+3 QC + confiance | script `implement_trust_framework.py` si sorties absentes | Notebooks 2 et 3 |
| 4 Figures mémoire | exécution du **Notebook 4** | — |
| 5 Contrôle | vérification des fichiers | `Notebook_0_Executer_pipeline_pas_a_pas.ipynb` |

> Les scripts sont préférés pour les gros volumes (~2,4 M lignes).  
> Les notebooks restent la référence méthodologique du mémoire.


## 1. Configuration


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import time

from IPython.display import display, Markdown

ROOT = Path.cwd()
DATA = ROOT / "data"
RAW = DATA / "raw"
PROCESSED = DATA / "processed"
METADATA = DATA / "metadata"
REPORTS = ROOT / "reports"
FIGURES = REPORTS / "figures"
TABLES = REPORTS / "tables"

# Forcer le recalcul même si le fichier existe déjà
FORCE_DOWNLOAD = False      # True = retélécharger openSenseMap
FORCE_QC_TRUST = False      # True = recalculer QC + confiance
FORCE_FIGURES = False       # True = régénérer tableaux/figures NB4

# Timeouts (secondes)
TIMEOUT_DOWNLOAD = 6 * 3600
TIMEOUT_QC_TRUST = 2 * 3600
TIMEOUT_NOTEBOOK = 30 * 60

PY = sys.executable

print("Racine :", ROOT)
print("Python :", PY)
print("FORCE_DOWNLOAD =", FORCE_DOWNLOAD)
print("FORCE_QC_TRUST =", FORCE_QC_TRUST)
print("FORCE_FIGURES  =", FORCE_FIGURES)


## 2. Fonctions utilitaires


In [ ]:
def exists(path: Path) -> bool:
    return path.exists()

def file_info(path: Path) -> str:
    if not path.exists():
        return "ABSENT"
    if path.is_dir():
        n = len(list(path.glob("*")))
        return f"dossier ({n} éléments)"
    return f"{path.stat().st_size / 1e6:.1f} Mo"

def run_script(script_name: str, timeout: int) -> None:
    script = ROOT / script_name
    if not script.exists():
        raise FileNotFoundError(f"Script introuvable : {script}")
    print(f"→ python {script_name}")
    t0 = time.time()
    subprocess.check_call(
        [PY, "-u", str(script)],
        cwd=str(ROOT),
        timeout=timeout,
    )
    print(f"   terminé en {time.time() - t0:.1f} s")

def run_notebook(notebook_name: str, timeout: int) -> None:
    import nbformat
    from nbclient import NotebookClient

    path = ROOT / notebook_name
    if not path.exists():
        raise FileNotFoundError(f"Notebook introuvable : {path}")
    print(f"→ exécution notebook : {notebook_name}")
    t0 = time.time()
    nb = nbformat.read(path, as_version=4)
    client = NotebookClient(
        nb,
        timeout=timeout,
        kernel_name="python3",
        resources={"metadata": {"path": str(ROOT)}},
    )
    client.execute()
    nbformat.write(nb, path)
    print(f"   terminé en {time.time() - t0:.1f} s")

print("Helpers prêts.")


## 3. État initial des sorties


In [ ]:
checkpoints = {
    "1. Raw NB1": RAW / "opensensemap_raw_long_active_50.csv",
    "1. Stations": METADATA / "stations_selectionnees_actives.csv",
    "2. Métriques QC": PROCESSED / "opensensemap_qc_metrics_station_variable.csv",
    "2. QC flagged": PROCESSED / "opensensemap_measurements_qc_flagged.csv",
    "3. Trust séries": PROCESSED / "opensensemap_trust_scores_series.csv",
    "3. Trust stations": PROCESSED / "opensensemap_trust_scores_stations.csv",
    "3. FFP summary": PROCESSED / "opensensemap_fit_for_purpose_summary.csv",
    "4. Figures": FIGURES / "fig1_distribution_trust_scores.png",
    "4. Tables": TABLES / "table1_synthese_corpus.csv",
}

print("=== ÉTAT INITIAL ===")
for label, path in checkpoints.items():
    status = "OK" if exists(path) else "--"
    print(f"[{status}] {label:18s} | {file_info(path)} | {path.name}")


## 4. Étape 1 — Acquisition (NB1)

Si le CSV brut existe déjà, l’étape est **sautée** (sauf `FORCE_DOWNLOAD=True`).

Alternative manuelle : ouvrir `Notebook_1_Acquisition_dataset.ipynb`.


In [ ]:
raw_csv = RAW / "opensensemap_raw_long_active_50.csv"
stations_csv = METADATA / "stations_selectionnees_actives.csv"

need_download = FORCE_DOWNLOAD or (not raw_csv.exists())

if not need_download:
    print("Étape 1 SKIP — dataset brut déjà présent")
    print(" ", file_info(raw_csv), "→", raw_csv.name)
    if stations_csv.exists():
        print(" ", file_info(stations_csv), "→", stations_csv.name)
else:
    print("Étape 1 RUN — téléchargement openSenseMap (peut être long)")
    run_script("download_opensensemap_thp.py", timeout=TIMEOUT_DOWNLOAD)
    assert raw_csv.exists(), "Échec : CSV brut toujours absent"

print("Étape 1 terminée.")


## 5. Étape 2+3 — Contrôle qualité + confiance (NB2 + NB3)

Un seul script produit les sorties QC **et** les scores de confiance  
(`implement_trust_framework.py`), aligné sur les Notebooks 2 et 3.

Sauté si métriques + scores existent déjà (sauf `FORCE_QC_TRUST=True`).


In [ ]:
metrics_csv = PROCESSED / "opensensemap_qc_metrics_station_variable.csv"
trust_csv = PROCESSED / "opensensemap_trust_scores_series.csv"

need_qc = FORCE_QC_TRUST or (not metrics_csv.exists()) or (not trust_csv.exists())

if not need_qc:
    print("Étape 2+3 SKIP — QC et confiance déjà calculés")
    print(" ", file_info(metrics_csv))
    print(" ", file_info(trust_csv))
else:
    if not (RAW / "opensensemap_raw_long_active_50.csv").exists():
        raise FileNotFoundError(
            "Dataset brut absent. Exécuter d'abord l'étape 1."
        )
    print("Étape 2+3 RUN — QC + confiance (peut prendre plusieurs minutes)")
    run_script("implement_trust_framework.py", timeout=TIMEOUT_QC_TRUST)
    assert metrics_csv.exists() and trust_csv.exists(), "Échec QC/confiance"

print("Étape 2+3 terminée.")


## 6. Étape 4 — Tableaux et figures mémoire (NB4)

Exécute **cellule après cellule** le Notebook 4  
(`Notebook_4_Resultats_memoire.ipynb`).

Produits : `reports/tables/`, `reports/figures/` (classes FFP, profils, cartes…).


In [ ]:
fig1 = FIGURES / "fig1_distribution_trust_scores.png"
nb4_name = "Notebook_4_Resultats_memoire.ipynb"

need_figures = FORCE_FIGURES or (not fig1.exists())

if not need_figures:
    print("Étape 4 SKIP — figures déjà présentes")
    print(" ", file_info(fig1))
    n_fig = len(list(FIGURES.glob("*.png")))
    n_tab = len(list(TABLES.glob("*.csv")))
    print(f"   {n_fig} figures | {n_tab} tableaux")
else:
    if not (PROCESSED / "opensensemap_trust_scores_series.csv").exists():
        raise FileNotFoundError(
            "Scores de confiance absents. Exécuter d'abord l'étape 2+3."
        )
    print("Étape 4 RUN — exécution du Notebook 4")
    run_notebook(nb4_name, timeout=TIMEOUT_NOTEBOOK)
    assert fig1.exists(), "Échec : figure principale absente après NB4"

print("Étape 4 terminée.")
print("Figures :", FIGURES)
print("Tableaux:", TABLES)


## 7. Étape 5 — Contrôle de cohérence final

Vérifie que tous les livrables obligatoires du pipeline sont présents.


In [ ]:
required = {
    "NB1 raw": RAW / "opensensemap_raw_long_active_50.csv",
    "NB1 stations": METADATA / "stations_selectionnees_actives.csv",
    "NB2 métriques": PROCESSED / "opensensemap_qc_metrics_station_variable.csv",
    "NB2 params QC": PROCESSED / "opensensemap_qc_parameters.json",
    "NB3 trust séries": PROCESSED / "opensensemap_trust_scores_series.csv",
    "NB3 trust stations": PROCESSED / "opensensemap_trust_scores_stations.csv",
    "NB3 FFP": PROCESSED / "opensensemap_fit_for_purpose_summary.csv",
    "NB3 params trust": PROCESSED / "opensensemap_trust_parameters.json",
    "NB4 fig1": FIGURES / "fig1_distribution_trust_scores.png",
    "NB4 table1": TABLES / "table1_synthese_corpus.csv",
    "NB4 manifeste": REPORTS / "manifest_memoires_exports.csv",
}

print("=== CONTRÔLE FINAL ===")
missing = []
for label, path in required.items():
    ok = exists(path)
    print(f"[{'OK' if ok else 'MANQUANT'}] {label:18s} | {file_info(path)}")
    if not ok:
        missing.append(str(path))

optional_figs = sorted(FIGURES.glob("fig*.png"))
print(f"\nFigures mémoire : {len(optional_figs)}")
for p in optional_figs:
    print("  -", p.name)

if missing:
    raise FileNotFoundError(
        "Pipeline incomplet. Fichiers manquants :\n- " + "\n- ".join(missing)
    )

display(Markdown("**Pipeline complet : NB1 → NB2 → NB3 → NB4 OK.**"))


## 8. Aperçu rapide des résultats (optionnel)


In [ ]:
import pandas as pd

trust = pd.read_csv(PROCESSED / "opensensemap_trust_scores_series.csv")
ffp = pd.read_csv(PROCESSED / "opensensemap_fit_for_purpose_summary.csv")
corpus = pd.read_csv(TABLES / "table1_synthese_corpus.csv")

display(Markdown("### Synthèse corpus"))
display(corpus)

display(Markdown("### Classes fit-for-purpose (séries)"))
display(ffp)

print(
    f"Score médian : {trust['trust_score'].median():.3f} | "
    f"séries : {len(trust)} | "
    f"stations : {trust['station_id'].nunique()}"
)


## 9. Bilan

Ce notebook a enchaîné le pipeline **cellule après cellule**.

### Où aller ensuite

| Besoin | Fichier |
|---|---|
| Méthode d’acquisition | `Notebook_1_Acquisition_dataset.ipynb` |
| Détail des drapeaux QC | `Notebook_2_Preparation_Controle_Qualite_openSenseMap.ipynb` |
| Détail scores / FFP | `Notebook_3_Score_confiance_fit_for_purpose.ipynb` |
| Tableaux & figures | `Notebook 4 Résultats...ipynb` / `reports/` |
| Contrôle seul | `Notebook_0_Executer_pipeline_pas_a_pas.ipynb` |

### Recalcul forcé

Dans la cellule de configuration :
- `FORCE_DOWNLOAD = True`
- `FORCE_QC_TRUST = True`
- `FORCE_FIGURES = True`

puis relancer les cellules concernées (ou `Run All`).
